In [4]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np  

The CNN is implmented using Pytorch's nn.Module and is designed to classify Mel spectograms into three diffrent categories: good, heavy load or broken -engine.

the arcutecture of the modle can be configured allowing one to chose the number of layers, number of kernals and the kernels size.

Convelutonal layers:
the convelutional layers are created and initelized dynamically using a for loop and stored in the Sequential container called layers. 

Each convelutional block consists of:
nn.Conv2d: Wich Extracts features from a Mel spectogram using lernable convelutional kernals with a padding set to 1. 

nn.ReLU: Is an activation function that introduces non-linearity, meaning it outputs the input directly if it is positive and outputs zero if it is negative or equal to zero. This allows the neural network to learn more complex patterns.

nn.MaxPool2d(2): It reduces the spatial dimensions of the featrue maps using a 2x2 matrix, selecting the maximum value from each region. This reduces the computation time while preserving the most prominent featrues. It can also help reduce overfitting.

The number of kernals doubles after each convelutional block, allowing deeper layers to learn more featrue representations, but can lead to overfitting.

Clasification Layer

at the end of the Sequential container/after the convelutional blocks, the extracted feature maps are passed through two layers.

nn.Flatten: Converts the feature maps into a one-dimensional feature vector for each sample, allowing the data to be passed into a fully connected layer.

mm.LazyLinear(3): is a fully connected layer that automatically determins the input size of the feature vector being passed to it and produces three output scores corrsponding to the three diffrent engin conditions.

Forward Propagation

The forward() methode defines how the input is being passed thourge the network. Since all the layers have been stored in a nn.sequential container the input will be processed sequrntily through the convelutional blocks and classification layers.

The CNN returns three raw output scores, which is used during the training loop with cross entropy loss function. 






In [5]:
class CNN(nn.Module):
 def __init__(self, numberOfLayers = 3, numberOfKernals = 16, kernelSize = 3, inChannels = 1):
  
  super(). __init__()
  self.layers = nn.Sequential()

  for i in range(numberOfLayers):
   self.layers.append(nn.Conv2d(inChannels, numberOfKernals, kernelSize, padding=1))
   self.layers.append(nn.ReLU())
   self.layers.append(nn.MaxPool2d(2))
   inChannels = numberOfKernals
   numberOfKernals *= 2 

  self.layers.append(nn.Flatten())
  self.layers.append(nn.LazyLinear(3))

 def forward(self, input):
  return self.layers(input)
  


In [6]:
melTrainingData = np.load("features/log_M_train.npy")
trainingLables = np.load("features/y_train.npy") 

In [7]:
input = torch.tensor(melTrainingData, dtype = torch.float32)
input = input.unsqueeze(1)

answer = torch.tensor(trainingLables, dtype = torch.long)

In [8]:
dataset = TensorDataset(input, answer)

trainLoader = DataLoader(dataset, batch_size=16, shuffle=True)

CNN Training

The Training() function is responsible for training the modle. It takes 4 parameters:

CNNmodel: The nurel network that will be trained.

trainLoader: A Dataloder that provides batches of mel spectogram and there corosponging labels 

lr: the leraning rate, which controls how much the modles wights are adjusted during training

epochs: The number of times the model processes the enteire dataset.

Loss Function and Optimizer

The model uses CrossEntropyLoss as its loss function to massure the diffrences between the CNNs predictions and the actuel class lables. It penelizes inncorrect predictions and helps the CNN improve its accuracy during traing through backpropgation. It is also importent to mention that the CrossEntropyLoss takes raw output and not probabilities it internerly applies LogSoftmax to obtain log-probabilities. These are then used to calculate the loss based on the correct labels.

The AdamW optimizer algorithem is used to update the wights of the neural network. Insted of weight decay and gradient update happening at the same time AdamW saparatetes them and applies wight decay as a separate operation after calculating the adaptive learning rate update step. This prevents the modles wights from growing too large which can help reduce overfitting and improve generalization.

Training loop

The traing process consists of two for loops. The outer loop controls the number og epochs, while the inner for loop processes the data in batches.

for each batch, the following operation happens.

Resets gradients: optimizer.zero_grad() resets the gradiants from prevouis iteration. This prevents the gradiants from last iteration to effect the new gradients. 

Forward prop.: outputs = CNNmodel(inputs) passes the Mel spectogram though the CNN to produce predictions. 

Calculate loss: criterion(outputs, labels) compared the predictions with the correct labels.

backprop.: loss.backward() calculates the gradiants for the models trainable parameters.

Update weights: optimizer.step() updates the parameters based on the calculated gradiants.


In [9]:
def Training(CNNmodel, trainLoader, lr = 0.001, epochs = 20):
  criterion = nn.CrossEntropyLoss()
  optimizer = torch.optim.AdamW(CNNmodel.parameters(), lr)



  for epoch in range(epochs):
   CNNmodel.train()
   total_loss = 0

   for inputs, labels in trainLoader:

      optimizer.zero_grad()
      outputs = CNNmodel(inputs)

      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()

      total_loss += loss.item() * inputs.size(0)

   avg_loss = total_loss / len(dataset)

   print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.10f}")

In [10]:
CNNmodel = CNN()
with torch.no_grad():CNNmodel(input[:1]) 

In [11]:
Training(CNNmodel, trainLoader)

Epoch 1/20, Loss: 0.1490702443
Epoch 2/20, Loss: 0.0000000220
Epoch 3/20, Loss: 0.0000000000
Epoch 4/20, Loss: 0.0000000000
Epoch 5/20, Loss: 0.0000000000
Epoch 6/20, Loss: 0.0000000000
Epoch 7/20, Loss: 0.0000000000
Epoch 8/20, Loss: 0.0000000000
Epoch 9/20, Loss: 0.0000000000
Epoch 10/20, Loss: 0.0000000000
Epoch 11/20, Loss: 0.0000000000
Epoch 12/20, Loss: 0.0000000000
Epoch 13/20, Loss: 0.0000000000
Epoch 14/20, Loss: 0.0000000000
Epoch 15/20, Loss: 0.0000000000
Epoch 16/20, Loss: 0.0000000000
Epoch 17/20, Loss: 0.0000000000
Epoch 18/20, Loss: 0.0000000000
Epoch 19/20, Loss: 0.0000000000
Epoch 20/20, Loss: 0.0000000000


In [17]:
CNNmodel.eval()

with torch.no_grad():
    outputs = CNNmodel(input) 
    predictions = outputs.argmax(dim=1)

    accuracy = (predictions == answer).float().mean()

print("Training accuracy:", accuracy.item())

Training accuracy: 1.0


In [13]:
melValidationData = np.load("features/log_M_val.npy")
validationLabels = np.load("features/y_val.npy")

In [14]:
valInput = torch.tensor(melValidationData, dtype=torch.float32)
valInput = valInput.unsqueeze(1)

valLabels = torch.tensor(validationLabels, dtype=torch.long)

the validation process is used to see how well the modle preformes to unseen data. This helps us determin whether the modle has learnt usful pattern or has overfitted to the training data.

Before validation CNNmodel.eval() is used to turn off layers that behave diffrently during traing and evaluation sush as dropout and batch normalization, currently this modle dosent use any of them but i have learnd that it is good practice hens why it is their. (this line can be removed)

the validation happens inside torch.no_grad(), which disables gradinat calculation. Becuse we dont intend to update the modles weights just evaluate it.

Forward Pass and Predictions

The validation mel spectograms are passed through the CNN using CNNmodel(valInput).

The argmax(dim=1) operation selects the class with the highst output score for each spectogram. These predictions are then compared with the correct validation labels.





In [16]:
CNNmodel.eval()

with torch.no_grad():
    outputs = CNNmodel(valInput)
    predictions = outputs.argmax(dim=1)

    accuracy = (predictions == valLabels).float().mean()

    loss = nn.CrossEntropyLoss()(outputs, valLabels)

print("Validation accuracy:", accuracy.item())
print("Validation loss:", loss.item())

Validation accuracy: 1.0
Validation loss: 0.0001646509044803679


In [374]:
torch.save({"model_state_dict": CNNmodel.state_dict(),"numberOfLayers": 3,"numberOfKernals": 16,"kernelSize": 3}, "CocainBot.pth")

USE THIS

In [17]:
import torch
from torch import nn
import numpy as np  


class CNN(nn.Module):
 def __init__(self, numberOfLayers = 3, numberOfKernals = 16, kernelSize = 3, inChannels = 1):
  
  super(). __init__()
  self.layers = nn.Sequential()

  for i in range(numberOfLayers):
   self.layers.append(nn.Conv2d(inChannels, numberOfKernals, kernelSize, padding=1))
   self.layers.append(nn.ReLU())
   self.layers.append(nn.MaxPool2d(2))
   inChannels = numberOfKernals
   numberOfKernals *= 2 

  self.layers.append(nn.Flatten())
  self.layers.append(nn.LazyLinear(3))

 def forward(self, input):
  return self.layers(input)


def loadModel(path):
    loadedModel = torch.load(path, map_location="cpu", weights_only=True)

    model = CNN(numberOfLayers = loadedModel["numberOfLayers"], numberOfKernals = loadedModel["numberOfKernals"], kernelSize = loadedModel["kernelSize"])

    model.load_state_dict(loadedModel["model_state_dict"])

    def Predict(data):
        input = torch.as_tensor(data, dtype=torch.float32)
        input = input.unsqueeze(1)

        with torch.no_grad():
          outputs = model(input)
          predictions = outputs.argmax(dim = 1)

        return predictions.numpy()
        
    return Predict

example

In [18]:
Predict = loadModel("CocainBot.pth")



In [ ]:
data = np.load("features/log_M_val.npy")

Predict(data[0:1])

array([0])